In [1]:
pip install transformers datasets torch scikit-learn tqdm


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\Camelin_NLP_4TIE\nlp_env\Scripts\python.exe -m pip install --upgrade pip' command.


In [2]:
import os
import json
import torch
from tqdm import tqdm
from transformers import BertTokenizer, EncoderDecoderModel
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [3]:
def load_data(folder_path, limit=None):
    data = []
    files = os.listdir(folder_path)
    files = sorted(files)[:limit]  # Ambil sebanyak limit saja

    for file in tqdm(files, desc=f"Loading from {folder_path}"):
        with open(os.path.join(folder_path, file), 'r', encoding='utf-8') as f:
            raw = json.load(f)
            # Gabungkan token jadi kalimat
            article = " ".join([" ".join(sent) for sent in raw["clean_article"]])
            summary = " ".join([" ".join(sent) for sent in raw["clean_summary"]])
            data.append((article, summary))
    return data


In [4]:
train_data = load_data("dataset/train", limit=1600)
test_data = load_data("dataset/test", limit=200)

# Pisahkan input dan target
train_texts, train_summaries = zip(*train_data)
test_texts, test_summaries = zip(*test_data)


Loading from dataset/test: 100%|██████████| 200/200 [00:00<00:00, 19999.07it/s]


In [5]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("cahya/bert2bert-indonesian-summarization")
tokenizer.bos_token = tokenizer.cls_token
tokenizer.eos_token = tokenizer.sep_token


In [6]:
def encode_data(texts, summaries, max_input=512, max_output=128):
    inputs = tokenizer(list(texts), padding='max_length', truncation=True, max_length=max_input, return_tensors='pt')
    outputs = tokenizer(list(summaries), padding='max_length', truncation=True, max_length=max_output, return_tensors='pt')
    outputs['input_ids'][outputs['input_ids'] == tokenizer.pad_token_id] = -100  # ignore loss on padding
    return inputs.input_ids, inputs.attention_mask, outputs.input_ids

In [7]:
train_input_ids, train_attn_mask, train_labels = encode_data(train_texts, train_summaries)
test_input_ids, test_attn_mask, test_labels = encode_data(test_texts, test_summaries)


In [8]:
from transformers import EncoderDecoderModel

model = EncoderDecoderModel.from_pretrained("cahya/bert2bert-indonesian-summarization")
model = model.to(device)

In [9]:
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(train_input_ids, train_attn_mask, train_labels)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)


In [11]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
from torch.nn.utils import clip_grad_norm_

epochs = 3
model.train()

for epoch in range(epochs):
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids, attn_mask, labels = [b.to(device) for b in batch]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attn_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()
    
    print(f"Epoch {epoch+1} - Loss: {total_loss / len(train_loader):.4f}")


Epoch 1:   0%|          | 0/400 [00:00<?, ?it/s]d:\Camelin_NLP_4TIE\nlp_env\lib\site-packages\transformers\models\encoder_decoder\modeling_encoder_decoder.py:557: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than tensor.new_tensor(sourceTensor).
  decoder_attention_mask = decoder_input_ids.new_tensor(decoder_input_ids != self.config.pad_token_id)
d:\Camelin_NLP_4TIE\nlp_env\lib\site-packages\transformers\models\encoder_decoder\modeling_encoder_decoder.py:577: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING

In [ ]:
model.eval()

def generate_summary(texts, max_input=512, max_output=128):
    inputs = tokenizer(texts, return_tensors="pt", padding="max_length", truncation=True, max_length=max_input).to(device)
    summaries = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_length=max_output,
        num_beams=4,
        early_stopping=True
    )
    return [tokenizer.decode(s, skip_special_tokens=True) for s in summaries]


In [ ]:
from datasets import load_metric

rouge = load_metric("rouge")

preds = generate_summary(test_texts[:100])
refs = [" ".join(s) for s in test_summaries[:100]]

results = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
for key in results:
    print(f"{key}: {results[key].mid.fmeasure:.4f}")


In [ ]:
model.save_pretrained("bert2bert-indonesian-summarization-finetuned")
tokenizer.save_pretrained("bert2bert-indonesian-summarization-finetuned")
